In [ ]:
!pip install transformers datasets scikit-learn pandas

In [1]:
#roberta-base
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
import numpy as np
import json
!pip uninstall wandb -y

# Load label mapping
with open("labels.json") as f:
    label2id = json.load(f)
id2label = {v: k for k, v in label2id.items()}

# Load session-based splits
train_df = pd.read_csv("train_4class.csv")
val_df = pd.read_csv("val_4class.csv")
test_df = pd.read_csv("test_4class.csv")

# Map emotion column to ids
for df in [train_df, val_df, test_df]:
    df["emotion_id"] = df["emotion"].map(label2id)

# Tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Dataset class
class EmotionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        inputs = self.tokenizer(
            row["clean_text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": torch.tensor(row["emotion_id"])
        }

# Create dataset objects
train_dataset = EmotionDataset(train_df, tokenizer)
val_dataset = EmotionDataset(val_df, tokenizer)
test_dataset = EmotionDataset(test_df, tokenizer)

# Load model
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./roberta-results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01
)


# Accuracy metric
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    accuracy = (preds == labels).mean()
    return {"accuracy": accuracy}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

import os
os.environ["WANDB_DISABLED"] = "true"

# Train the model
trainer.train()

# Evaluate on test set
results = trainer.evaluate(test_dataset)
print("📊 Final Test Set Evaluation:")
print(results)


Found existing installation: wandb 0.22.3
Uninstalling wandb-0.22.3:
  Successfully uninstalled wandb-0.22.3


FileNotFoundError: [Errno 2] No such file or directory: 'labels.json'

In [ ]:
!pip install -U transformers

In [ ]:
#roberta-large
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
import numpy as np
import json
import os

# Disable W&B logging
os.environ["WANDB_DISABLED"] = "true"

# Load label mapping
with open("labels.json") as f:
    label2id = json.load(f)
id2label = {v: k for k, v in label2id.items()}

# Load data
train_df = pd.read_csv("train_4class.csv")
val_df = pd.read_csv("val_4class.csv")
test_df = pd.read_csv("test_4class.csv")

# Map emotion text to IDs
for df in [train_df, val_df, test_df]:
    df["emotion_id"] = df["emotion"].map(label2id)

# Load tokenizer for roberta-large
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Dataset class
class EmotionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        inputs = self.tokenizer(
            row["clean_text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": torch.tensor(row["emotion_id"])
        }

# Create dataset objects
train_dataset = EmotionDataset(train_df, tokenizer)
val_dataset = EmotionDataset(val_df, tokenizer)
test_dataset = EmotionDataset(test_df, tokenizer)

# Load model
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-large",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Training args
training_args = TrainingArguments(
    output_dir="./roberta_large_results",
    per_device_train_batch_size=8,  # roberta-large needs smaller batches
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",  # Optional: specify a logging directory
    logging_steps=10,  # log every 10 steps
    save_steps=500,  # Save model checkpoints every 500 steps
    eval_strategy="epoch",  # Default: evaluate every "steps" (you can set it to "epoch" if your version supports it)
    save_strategy="epoch", # Set save strategy to epoch
    save_total_limit=2  # Limit the number of saved models to avoid excessive storage use
)

# Accuracy metric
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    accuracy = (preds == labels).mean()
    return {"accuracy": accuracy}

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)


# Check if checkpoint exists
last_checkpoint = None
output_dir = training_args.output_dir
if os.path.isdir(output_dir):
    checkpoints = [os.path.join(output_dir, d) for d in os.listdir(output_dir) if d.startswith("checkpoint")]
    if checkpoints:
        last_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
        print(f"🛠️ Resuming from checkpoint: {last_checkpoint}")

# Train (resume if checkpoint found)
trainer.train(resume_from_checkpoint=last_checkpoint)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy
1,1.411400,1.438262,0.250755
2,1.410900,1.408021,0.302115


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,1.411400,1.438262,0.250755
2,1.410900,1.408021,0.302115
3,1.368400,1.401237,0.250755
4,1.345900,1.407750,0.250755
5,1.388800,1.428307,0.250755
6,1.387300,1.407143,0.250755


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


TrainOutput(global_step=2346, training_loss=1.3753317571153827, metrics={'train_runtime': 2003.9411, 'train_samples_per_second': 9.363, 'train_steps_per_second': 1.171, 'total_flos': 4371253598582784.0, 'train_loss': 1.3753317571153827, 'epoch': 6.0})

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
import numpy as np
import json
import os

# Load label mapping
with open("labels.json") as f:
    label2id = json.load(f)
id2label = {v: k for k, v in label2id.items()}

# Load data
train_df = pd.read_csv("train_4class.csv")
val_df = pd.read_csv("val_4class.csv")
test_df = pd.read_csv("test_4class.csv")

# Map emotion text to IDs
for df in [train_df, val_df, test_df]:
    df["emotion_id"] = df["emotion"].map(label2id)

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Dataset class
class EmotionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        inputs = self.tokenizer(
            row["clean_text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": torch.tensor(row["emotion_id"])
        }

# Dataset objects
train_dataset = EmotionDataset(train_df, tokenizer)
val_dataset = EmotionDataset(val_df, tokenizer)
test_dataset = EmotionDataset(test_df, tokenizer)

# Load BERT model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./bert_results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=2
)

# Accuracy metric
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    accuracy = (preds == labels).mean()
    return {"accuracy": accuracy}

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train
trainer.train()

# Evaluate
results = trainer.evaluate(test_dataset)
print("📊 Final Evaluation Results:")
print(results)
print(f"✅ Final Accuracy on Test Set: {results['eval_accuracy']:.4f}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,1.148000,1.101455,0.537764
2,0.734100,0.979409,0.626385
3,0.608100,1.099684,0.633434
4,0.293500,1.220098,0.637462
5,0.243300,1.378598,0.640483


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packa

📊 Final Evaluation Results:
{'eval_loss': 1.3536063432693481, 'eval_accuracy': 0.6296603148301574, 'eval_runtime': 8.3294, 'eval_samples_per_second': 144.908, 'eval_steps_per_second': 9.124, 'epoch': 5.0}
✅ Final Accuracy on Test Set: 0.6297


In [ ]:
pip install transformers==4.36.2 datasets scikit-learn


In [ ]:
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 96.8 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.36.2
    Uninstalling transformers-4.36.2:
      Successfully uninstalled transformers-4.36.2


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import DebertaV2Tokenizer, DebertaV2ForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
import numpy as np
import json
import os

# Disable Weights & Biases logging
os.environ["WANDB_DISABLED"] = "true"

# Load label mappings
with open("labels.json") as f:
    label2id = json.load(f)
id2label = {v: k for k, v in label2id.items()}

# Load datasets
train_df = pd.read_csv("train_4class.csv")
val_df = pd.read_csv("val_4class.csv")
test_df = pd.read_csv("test_4class.csv")

# Map emotion labels to IDs
for df in [train_df, val_df, test_df]:
    df["emotion_id"] = df["emotion"].map(label2id)

# Tokenizer
tokenizer = DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-base")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Custom Dataset
class EmotionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        inputs = self.tokenizer(
            row["clean_text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": torch.tensor(row["emotion_id"])
        }

# Dataset objects
train_dataset = EmotionDataset(train_df, tokenizer)
val_dataset = EmotionDataset(val_df, tokenizer)
test_dataset = EmotionDataset(test_df, tokenizer)

# Load model
model = DebertaV2ForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./deberta_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    save_total_limit=2
)

# Metric function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    accuracy = (preds == labels).mean()
    return {"accuracy": accuracy}

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train
trainer.train()

# Evaluate on test
results = trainer.evaluate(test_dataset)
print("📊 Final Accuracy on Test Set:", results["eval_accuracy"])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,1.372200,1.419359,0.250755
2,1.361000,1.387351,0.334340
3,1.296200,1.374436,0.365559


Epoch,Training Loss,Validation Loss,Accuracy
1,1.372200,1.419359,0.250755
2,1.361000,1.387351,0.334340
3,1.296200,1.374436,0.365559
4,1.213300,1.329757,0.377644
5,1.149400,1.392880,0.394763
6,1.094800,1.393121,0.392749


📊 Final Accuracy on Test Set: 0.4772162386081193


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import numpy as np
import json
import os

# Disable Weights & Biases logging
os.environ["WANDB_DISABLED"] = "true"

# Load label mapping
with open("labels.json") as f:
    label2id = json.load(f)
id2label = {v: k for k, v in label2id.items()}

# Load data
train_df = pd.read_csv("train_4class.csv")
val_df = pd.read_csv("val_4class.csv")
test_df = pd.read_csv("test_4class.csv")

# Map emotion text to IDs
for df in [train_df, val_df, test_df]:
    df["emotion_id"] = df["emotion"].map(label2id)

# Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Dataset class
class EmotionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        inputs = self.tokenizer(
            row["clean_text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": torch.tensor(row["emotion_id"])
        }

# Load datasets
train_dataset = EmotionDataset(train_df, tokenizer)
val_dataset = EmotionDataset(val_df, tokenizer)
test_dataset = EmotionDataset(test_df, tokenizer)

# Load model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Training Arguments
training_args = TrainingArguments(
    output_dir="./bert_results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    save_total_limit=2
)

# Accuracy metric
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    accuracy = (preds == labels).mean()
    return {"accuracy": accuracy}

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train
trainer.train()

# Evaluate
results = trainer.evaluate(test_dataset)
print("📊 Final Test Set Evaluation:")
print(results)
print(f"✅ Final Accuracy on Test Set: {round(results['eval_accuracy'] * 100, 2)}%")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,1.201700,1.055922,0.569990
2,0.803700,1.028276,0.601208
3,0.533300,1.067998,0.620342
4,0.373700,1.199363,0.625378
5,0.306900,1.367636,0.623364
6,0.255900,1.467332,0.634441
7,0.213900,1.666031,0.626385
8,0.181400,1.832975,0.624371
9,0.169200,1.816323,0.626385
10,0.147500,1.843282,0.618328


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packa

📊 Final Test Set Evaluation:
{'eval_loss': 1.4461318254470825, 'eval_accuracy': 0.628003314001657, 'eval_runtime': 8.7215, 'eval_samples_per_second': 138.394, 'eval_steps_per_second': 8.714, 'epoch': 10.0}
✅ Final Accuracy on Test Set: 62.8%


In [ ]:
!pip install -q sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.7 MB/s eta 0:00:00


In [ ]:
# ✅ 2. Imports
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# ✅ 3. Load CSVs
train_df = pd.read_csv("train_4class.csv")
val_df = pd.read_csv("val_4class.csv")
test_df = pd.read_csv("test_4class.csv")

# ✅ 4. Encode labels
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['emotion'])
val_df['label'] = le.transform(val_df['emotion'])
test_df['label'] = le.transform(test_df['emotion'])

# ✅ 5. Load pretrained Sentence-BERT model
model = SentenceTransformer("all-MiniLM-L6-v2")  # Lightweight and fast

# ✅ 6. Create embeddings
print("🔍 Embedding text...")
X_train = model.encode(train_df['clean_text'].tolist(), show_progress_bar=True, convert_to_numpy=True)
X_val = model.encode(val_df['clean_text'].tolist(), show_progress_bar=True, convert_to_numpy=True)
X_test = model.encode(test_df['clean_text'].tolist(), show_progress_bar=True, convert_to_numpy=True)

y_train = train_df['label']
y_val = val_df['label']
y_test = test_df['label']

# ✅ 7. Train logistic regression
print("⚙️ Training classifier...")
clf = LogisticRegression(max_iter=2000, solver='lbfgs', class_weight='balanced')
clf.fit(X_train, y_train)

# ✅ 8. Evaluation
print("\n📊 Validation Results:")
val_preds = clf.predict(X_val)
print(classification_report(y_val, val_preds, target_names=le.classes_))
print("🎯 Validation Accuracy:", accuracy_score(y_val, val_preds))

print("\n✅ Final Test Results:")
test_preds = clf.predict(X_test)
print(classification_report(y_test, test_preds, target_names=le.classes_))
print("🎯 Final Test Accuracy:", accuracy_score(y_test, test_preds))


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/98 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

⚙️ Training classifier...

📊 Validation Results:
              precision    recall  f1-score   support

         ang       0.68      0.54      0.60       307
         hap       0.63      0.62      0.62       300
         neu       0.48      0.45      0.46       249
         sad       0.39      0.64      0.49       137

    accuracy                           0.55       993
   macro avg       0.55      0.56      0.55       993
weighted avg       0.58      0.55      0.56       993

🎯 Validation Accuracy: 0.5548841893252769

✅ Final Test Results:
              precision    recall  f1-score   support

         ang       0.46      0.54      0.50       167
         hap       0.67      0.54      0.60       432
         neu       0.54      0.51      0.52       373
         sad       0.47      0.61      0.53       235

    accuracy                           0.55      1207
   macro avg       0.53      0.55      0.54      1207
weighted avg       0.56      0.55      0.55      1207

🎯 Final Test Acc

In [ ]:
# BERT-based Emotion Recognition on IEMOCAP (4-class)
# Colab-ready | Leave-One-Session-Out (LOSO) Cross Validation

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
import os

# Disable Weights & Biases logging
os.environ["WANDB_DISABLED"] = "true"

# Set seed for reproducibility
torch.manual_seed(42)

# Load data (must have 'clean_text', 'emotion', 'utterance_id')
df = pd.read_csv("iemocap_cleaned.csv")

# Reduce to 4 major emotions
target_emotions = ["ang", "hap", "sad", "neu"]
df = df[df["emotion"].isin(target_emotions)]

# Map to integer labels
label2id = {emo: idx for idx, emo in enumerate(target_emotions)}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df["emotion"].map(label2id)

# Extract session info (assumes 'Ses0X' in utterance_id)
df["session"] = df["utterance_id"].apply(lambda x: x.split("_")[0])
sessions = sorted(df["session"].unique())

# Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Dataset class
class IEMOCAPDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        inputs = self.tokenizer(
            row["clean_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": torch.tensor(row["label"], dtype=torch.long)
        }

# Loop through LOSO sessions
acc_list = []

for test_session in sessions:
    print(f"\n🚀 Testing on session: {test_session}")

    train_df = df[df["session"] != test_session].reset_index(drop=True)
    test_df = df[df["session"] == test_session].reset_index(drop=True)

    train_dataset = IEMOCAPDataset(train_df, tokenizer)
    test_dataset = IEMOCAPDataset(test_df, tokenizer)

    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=4,
        id2label=id2label,
        label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f"./bert_results_{test_session}",
        num_train_epochs=5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-5,
        logging_dir=f"./logs_{test_session}",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        save_total_limit=1,
    )

    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=1)
        acc = accuracy_score(p.label_ids, preds)
        return {"accuracy": acc}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_result = trainer.evaluate()
    acc = eval_result["eval_accuracy"]
    print(f"✅ Accuracy on {test_session}: {acc:.4f}")
    acc_list.append(acc)

# Final summary
print("\n📊 Average Accuracy (LOSO):", round(np.mean(acc_list), 4))